In [1]:
import cv2
import numpy as np
import os

# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = r"C:/Users/Asanthi Upeksha/Desktop/Segmentation_part/vid_1.MOV"

# Main Output Folder
MAIN_OUTPUT_FOLDER = r"C:/Users/Asanthi Upeksha/Desktop/Segmentation_Output"

MIN_AREA = 5000

# =====================================================
# CREATE SUBFOLDERS
# =====================================================

FRAME_FOLDER = os.path.join(MAIN_OUTPUT_FOLDER, "Frames")

ENHANCED_FOLDER = os.path.join(MAIN_OUTPUT_FOLDER, "Enhanced")

MASK_FOLDER = os.path.join(MAIN_OUTPUT_FOLDER, "Masks")

CONTOUR_FOLDER = os.path.join(MAIN_OUTPUT_FOLDER, "Contours")

# Create folders
os.makedirs(FRAME_FOLDER, exist_ok=True)

os.makedirs(ENHANCED_FOLDER, exist_ok=True)

os.makedirs(MASK_FOLDER, exist_ok=True)

os.makedirs(CONTOUR_FOLDER, exist_ok=True)


# =====================================================
# IMAGE ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Gaussian Blur
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Contrast Enhancement
    enhanced = cv2.equalizeHist(blur)

    # Sharpening
    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharpened = cv2.filter2D(enhanced, -1, kernel)

    return sharpened


# =====================================================
# POTHOLE SEGMENTATION
# =====================================================

def segment_pothole(image):

    # Otsu Thresholding
    _, thresh = cv2.threshold(
        image,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Morphological Operations
    kernel = np.ones((5, 5), np.uint8)

    # Remove Noise
    opening = cv2.morphologyEx(
        thresh,
        cv2.MORPH_OPEN,
        kernel,
        iterations=2
    )

    # Fill Gaps
    closing = cv2.morphologyEx(
        opening,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return closing


# =====================================================
# CONTOUR VISUALIZATION
# =====================================================

def draw_pothole_contours(frame, mask):

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    pothole_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > MIN_AREA:

            # Draw Contour
            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            # Bounding Box
            x, y, w, h = cv2.boundingRect(cnt)

            cv2.rectangle(
                output,
                (x, y),
                (x + w, y + h),
                (0, 0, 255),
                2
            )

            # Label
            cv2.putText(
                output,
                "Pothole",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

            pothole_count += 1

    return output


# =====================================================
# MAIN PROCESS
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():

        print("Error opening video")
        return

    frame_count = 0

    print("Starting segmentation...")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # =================================================
        # IMAGE ENHANCEMENT
        # =================================================

        enhanced = enhance_frame(frame)

        # =================================================
        # SEGMENTATION MASK
        # =================================================

        mask = segment_pothole(enhanced)

        # =================================================
        # CONTOUR OUTPUT
        # =================================================

        contour_output = draw_pothole_contours(
            frame,
            mask
        )

        # =================================================
        # SAVE OUTPUTS IN SEPARATE FOLDERS
        # =================================================

        # Original Frames
        cv2.imwrite(
            os.path.join(
                FRAME_FOLDER,
                f"frame_{frame_count}.jpg"
            ),
            frame
        )

        # Enhanced Images
        cv2.imwrite(
            os.path.join(
                ENHANCED_FOLDER,
                f"enhanced_{frame_count}.jpg"
            ),
            enhanced
        )

        # Mask Outputs
        cv2.imwrite(
            os.path.join(
                MASK_FOLDER,
                f"mask_{frame_count}.jpg"
            ),
            mask
        )

        # Contour Outputs
        cv2.imwrite(
            os.path.join(
                CONTOUR_FOLDER,
                f"contour_{frame_count}.jpg"
            ),
            contour_output
        )

        # =================================================
        # DISPLAY
        # =================================================

        cv2.imshow("Original Frame", frame)

        cv2.imshow("Enhanced", enhanced)

        cv2.imshow("Mask Output", mask)

        cv2.imshow("Contour Output", contour_output)

        frame_count += 1

        # Press Q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

    # =====================================================
    # RELEASE
    # =====================================================

    cap.release()

    cv2.destroyAllWindows()

    print("\nProcessing Completed")
    print("Total Frames:", frame_count)

    print("\nOutputs saved inside:")
    print(MAIN_OUTPUT_FOLDER)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation...

Processing Completed
Total Frames: 66

Outputs saved inside:
C:/Users/Asanthi Upeksha/Desktop/Segmentation_Output
